In [ ]:
# !pip install pytorch-tabnet

In [ ]:
# !pip install tab-transformer-pytorch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gc
import os
import sys

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

In [ ]:
## perform classification using XGBoost
def perform_classification(df, random_state):
    X = df.drop(columns=['label'])
    y = df['label']
    X = X.to_numpy()
    y = y.to_numpy()
    
    skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    model = XGBClassifier(device='cuda')
    y_pred = cross_val_predict(model, X, y, cv=skfold)
    y_proba = cross_val_predict(model, X, y, cv=skfold, method='predict_proba')[:, 1]  ## select the probs of class 1 only
    
    accuracy = accuracy_score(y, y_pred)
    auroc = roc_auc_score(y, y_proba)
    cm = confusion_matrix(y, y_pred)

    results = {
        'accuracy':accuracy,
        'auroc':auroc,
        'cm':cm,
        'y_pred': y_pred,
        'y_proba':y_proba,
        'y_true':y
    }
    return results

In [ ]:
def compute_fusion_results(results_cnv, results_rna):
    
    fusion_prob_class_0 = 0.
    fusion_prob_class_1 = 0.
    
    tp_soft = 0
    fp_soft = 0
    tn_soft = 0
    fn_soft = 0
    
    tp_hard = 0
    fp_hard = 0
    tn_hard = 0
    fn_hard = 0
    
    for y_true, prob_cnv, prob_rna, pred_cnv, pred_rna in zip(results_rna['y_true'], results_cnv['y_proba'], results_rna['y_proba'], results_cnv['y_pred'], results_rna['y_pred']):
        
        ## perform hard voting
        fused_pred = np.where(pred_rna + pred_cnv >= 1, 1, 0)
        if y_true == 1 and fused_pred == 1:
            tp_hard = tp_hard + 1
        if y_true == 1 and fused_pred == 0:
            fn_hard = fn_hard + 1
            
        if y_true == 0 and fused_pred == 0:
            tn_hard = tn_hard + 1
            
        if y_true == 0 and fused_pred == 1:
            fp_hard = fp_hard + 1

        
        fusion_prob_class_0 = ((1-prob_cnv) + (1-prob_rna))/2
        fusion_prob_class_1 = ((prob_cnv) + (prob_rna))/2
    
        if y_true == 1 and (fusion_prob_class_1 > fusion_prob_class_0):
            tp_soft = tp_soft + 1
        if y_true == 1 and (fusion_prob_class_1 < fusion_prob_class_0):
            fn_soft = fn_soft + 1
            
        if y_true == 0 and (fusion_prob_class_0 > fusion_prob_class_1):
            tn_soft = tn_soft + 1
            
        if y_true == 0 and (fusion_prob_class_0 < fusion_prob_class_1):
            fp_soft = fp_soft + 1

    soft_vote = {
        'TP':tp_soft,
        'FP':fp_soft,
        'TN':tn_soft,
        'FN':fn_soft
    }
    hard_vote = {
        'TP':tp_hard,
        'FP':fp_hard,
        'TN':tn_hard,
        'FN':fn_hard
    }
    return soft_vote, hard_vote

In [ ]:
def dump_all_results(soft_vote, hard_vote, seed):
    
    acc_soft = (soft_vote['TP']+soft_vote['TN'])/(soft_vote['TP']+soft_vote['TN']+soft_vote['FP']+soft_vote['FN'])
    acc_hard = (hard_vote['TP']+hard_vote['TN'])/(hard_vote['TP']+hard_vote['TN']+hard_vote['FP']+hard_vote['FN'])
    print
    # Create results DataFrame
    results_df = pd.DataFrame({
        "acc_soft": [acc_soft],
        'Soft_TP':[soft_vote['TP']],
        'Soft_TN':[soft_vote['TN']],
        'Soft_FP':[soft_vote['FP']],
        'Soft_FN':[soft_vote['FN']],
        "acc_hard": [acc_hard],
        'Hard_TP':[hard_vote['TP']],
        'Hard_TN':[hard_vote['TN']],
        'Hard_FP':[hard_vote['FP']],
        'Hard_FN':[hard_vote['FN']],
        
    })

    print(f'Acc_soft:\t{acc_soft} \t\t Acc_hard:\t{acc_hard}')
    results_df.to_csv(f"Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/training_cohort/RESULTS/SOFT_AND_HARD_VOTING/{seed}.csv", index=False) ## Print results

In [ ]:
rna_luad_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\gene_exp\with_common_patients_processed', 'csv_rna_common_luad.csv'))
rna_lusc_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\gene_exp\with_common_patients_processed', 'csv_rna_common_lusc.csv'))

rna_luad_xena['label'] = 1
rna_lusc_xena['label'] = 0
df_rna_xena = pd.concat([rna_luad_xena, rna_lusc_xena], axis=0)

cnv_luad_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\cnv\with_common_patients_processed', 'csv_cnv_common_luad.csv'))
cnv_lusc_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\cnv\with_common_patients_processed', 'csv_cnv_common_lusc.csv'))

cnv_luad_xena['label'] = 1
cnv_lusc_xena['label'] = 0
df_cnv_xena = pd.concat([cnv_luad_xena, cnv_lusc_xena], axis=0)

In [ ]:
## sample with same seed, else samples would shuffle and then fusing of probs will not be correct
for seed in range(5):
    _df_rna_xena = df_rna_xena.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    _df_cnv_xena = df_cnv_xena.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    
    ## get XGBoost results on RNASeq
    results_rna = perform_classification(_df_rna_xena, seed)
    
    ## get XGBoost results on CNV
    results_cnv = perform_classification(_df_cnv_xena, seed)
    
    soft_vote, hard_vote = compute_fusion_results(results_cnv, results_rna)
    
    dump_all_results(soft_vote, hard_vote, seed)